# Stage 14 — Independent final-test evaluation (yeniden oluşturulan notebook)

**İki aşamalı, leakage-safe protokol.** İlk çalıştırmada `RUN_EVALUATION = False` kalmalı. Eski `validated_v007` / CLEAN_DEV (970 soru/model) üzerinde Logistic CORE ve XGBoost CORE yeniden fit edilir; 250 yeni soru için tahminler **test doğruluk etiketleri yüklenmeden** SHA ile dondurulur. İkinci aşamada yalnız son hücrede `RUN_EVALUATION=True` yapılarak 750 dondurulmuş tahmin değerlendirilir.

Stage 09–13 veya dondurulmuş diğer dosyalar yeniden çalıştırılmaz. Bu notebook Stage 12 v2 protokolünü ve Stage 13 zorluk-kotası değişikliğini doğrular, fakat onları değiştirmez. İlk aşamada final test model matrisinin yalnız özellik/anahtar sütunlarını okur; `manual_is_correct` sütununu ikinci aşamaya kadar yüklemez.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
%pip -q install 'numpy>=1.26,<3' 'pandas>=2.0,<3' 'scikit-learn>=1.5,<1.9' 'xgboost>=2.0,<3.2' 'pyarrow>=15,<25'

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.5/351.5 MB 3.7 MB/s eta 0:00:00


## 1 — Sabit sürümler, dosya yolları ve SHA yardımcıları

In [2]:
from pathlib import Path
import hashlib, json, os, math
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score, average_precision_score, log_loss
from xgboost import XGBClassifier

ROOT = Path('/content/drive/MyDrive/LLM_Confidence_Thesis')
DEV_VERSION = 'validated_v007'
TEST_VERSION = 'validated_v012_final250_v1'
PROTOCOL_PATH = ROOT/'results'/'stage12'/DEV_VERSION/'final_test_protocol_v2.json'
M12_PATH = ROOT/'results'/'stage12'/DEV_VERSION/'stage12_protocol_manifest_v2.json'
S13_DIR = ROOT/'results'/'stage13'/TEST_VERSION
M13_PATH = S13_DIR/'stage13_final_test_manifest_v1.json'
AMEND_PATH = S13_DIR/'stage12_v2_difficulty_amendment_v1.json'
QUESTIONS_PATH = S13_DIR/'final_test_questions_v1.json'
SPLIT_PATH = S13_DIR/'candidate_split_registry_v1.json'
DEV_SCHEMA_PATH = ROOT/'features'/'assembled'/DEV_VERSION/'feature_schema_v1.json'
DEV_M08_PATH = ROOT/'features'/'assembled'/DEV_VERSION/'stage08_manifest_v2_repair1.json'
TEST_SCHEMA_PATH = ROOT/'features'/'assembled'/TEST_VERSION/'feature_schema_v1.json'
TEST_M08_PATH = ROOT/'features'/'assembled'/TEST_VERSION/'stage08_manifest_v1.json'
TEST_MATRIX_PATH = ROOT/'features'/'assembled'/TEST_VERSION/'core_model_matrix_v1.parquet'
TEST_DATA_PATH = ROOT/'data'/'validated'/f'{TEST_VERSION}.json'
TEST_DATA_MANIFEST_PATH = ROOT/'data'/'validated'/f'{TEST_VERSION}_manifest.json'
OLD_DEV_PATH = ROOT/'data'/'validated'/'validated_v007.json'
OUT = ROOT/'results'/'stage14'/TEST_VERSION
PRED_PATH = OUT/'final_test_predictions_v1.parquet'
PRED_MANIFEST_PATH = OUT/'stage14_predictions_manifest_v1.json'
EVAL_METRICS_PATH = OUT/'final_test_metrics_v1.csv'
EVAL_SELECTIVE_PATH = OUT/'final_test_selective_v1.csv'
EVAL_BOOTSTRAP_PATH = OUT/'final_test_paired_bootstrap_v1.csv'
EVAL_MANIFEST_PATH = OUT/'stage14_evaluation_manifest_v1.json'

def require(condition, message):
    if not condition: raise RuntimeError(message)
def digest(path):
    path=Path(path)
    require(path.is_file(), f'Missing file: {path}')
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()
def load_json(path):
    with Path(path).open(encoding='utf-8') as f:return json.load(f)
def immutable_bytes(path, blob):
    path=Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        require(path.read_bytes()==blob, f'Immutable artifact conflict: {path}')
        return digest(path)
    temp=path.with_name(path.name+'.tmp')
    with temp.open('xb') as f:f.write(blob)
    os.replace(temp,path)
    return digest(path)
def freeze_json(path, obj):
    return immutable_bytes(path,json.dumps(obj,sort_keys=True,indent=2,ensure_ascii=False,allow_nan=False).encode('utf-8'))
def freeze_csv(path, frame):
    return immutable_bytes(path,frame.to_csv(index=False,na_rep='',float_format='%.12g').encode('utf-8'))
def freeze_parquet(path, frame):
    path=Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        existing=pd.read_parquet(path)
        pd.testing.assert_frame_equal(existing.reset_index(drop=True),frame.reset_index(drop=True),check_exact=True)
        return digest(path)
    temp=path.with_name(path.name+'.tmp.parquet')
    require(not temp.exists(),f'Stale temp file: {temp}')
    frame.to_parquet(temp,index=False,engine='pyarrow')
    os.replace(temp,path)
    return digest(path)

RUN_EVALUATION=False  # Son değerlendirme hücresinde ayrı değiştirilecek; şimdilik False.
print('Stage14 first pass: freeze predictions before opening test labels')

Stage14 first pass: freeze predictions before opening test labels


## 2 — Stage 12/13 ve yeni Stage 08 bütünlük kontrolleri (test etiketi yüklenmez)

In [3]:
p12,m12,m13,amend = map(load_json,[PROTOCOL_PATH,M12_PATH,M13_PATH,AMEND_PATH])
require(m12['protocol_sha256']==digest(PROTOCOL_PATH),'Stage12 protocol SHA')
require(p12['protocol_version']=='final_test_protocol_v2','Stage12 protocol version')
require(m13['stage12_protocol_sha256']==digest(PROTOCOL_PATH),'Stage13 Stage12 link')
require(m13['parent_validated_sha256']==digest(m13['parent_validated_dataset']),'Stage13 validated_v012 SHA')
require(m13['old_dev_validated_sha256']==digest(OLD_DEV_PATH),'Stage13 validated_v007 SHA')
require(amend['original_stage12_protocol_sha256']==digest(PROTOCOL_PATH),'Amendment Stage12 SHA')
require(amend['difficulty_selection_rule']=='descriptive_only','Unexpected difficulty amendment')
require(m13.get('difficulty_quotas_enforced') is False,'Difficulty quota amendment missing from Stage13 manifest')
require(m13['selection_protocol_amendment_sha256']==digest(AMEND_PATH),'Stage13 amendment SHA')
require(m13['final_test_questions_sha256']==digest(QUESTIONS_PATH),'Stage13 selected questions SHA')
require(m13['final_test_gold_sha256']==digest(m13['final_test_gold_path']),'Stage13 frozen gold SHA')
require(m13['split_registry_sha256']==digest(SPLIT_PATH),'Stage13 split SHA')
require(m13['inference_dataset_sha256']==digest(TEST_DATA_PATH),'Stage13 inference dataset SHA')
require(m13['inference_dataset_manifest_sha256']==digest(TEST_DATA_MANIFEST_PATH),'Stage13 derived manifest SHA')
require(m13['candidate_count']==300 and m13['n_final_questions']==250 and m13['n_reserve_questions']==50,'Stage13 counts')
require(m13['status']=='FROZEN_PREINFERENCE','Stage13 not frozen')
qs=load_json(QUESTIONS_PATH); split=load_json(SPLIT_PATH)
selected_ids=set(m13['final_question_ids'])
require(len(selected_ids)==len(qs)==250,'Selected question count')
require(selected_ids=={r['question_id'] for r in qs},'Selected question IDs mismatch')
require(sum(r['dataset_split']=='FINAL_TEST' for r in split)==250,'Split FINAL_TEST count')
require(sum(r['dataset_split']=='RESERVE' for r in split)==50,'Split reserve count')
require(selected_ids=={r['question_id'] for r in split if r['dataset_split']=='FINAL_TEST'},'Split IDs mismatch')
require(p12['final_test_registry']['n_independent_questions']==250,'Stage12 n final')
require(m13['category_quotas']==p12['final_test_registry']['category_final_quotas'],'Category quota mismatch')
require(m13['answer_type_quotas']==p12['final_test_registry']['answer_type_final_quotas'],'Answer type quota mismatch')
require({k:sum(r['category']==k for r in qs) for k in m13['category_quotas']}==m13['category_quotas'],'Selected category count mismatch')
require({k:sum(r['answer_type']==k for r in qs) for k in m13['answer_type_quotas']}==m13['answer_type_quotas'],'Selected answer-type count mismatch')

m08dev=load_json(DEV_M08_PATH); m08test=load_json(TEST_M08_PATH)
sdev=load_json(DEV_SCHEMA_PATH); stest=load_json(TEST_SCHEMA_PATH)
require(digest(DEV_M08_PATH)==p12['stage08_manifest_sha256'],'Frozen development Stage08 SHA')
require(digest(DEV_SCHEMA_PATH)==p12['stage08_schema_sha256'],'Frozen development schema SHA')
require(digest(TEST_SCHEMA_PATH)==m08test['outputs']['feature_schema_file_sha256'],'New schema file SHA')
require(digest(TEST_MATRIX_PATH)==m08test['outputs']['core_model_matrix_sha256'],'New core matrix SHA')
require(m08test['dataset_version']==TEST_VERSION and m08test['n_questions']==1250 and m08test['n_model_answer_rows']==3750,'New Stage08 scope')
require(m08test['upstream']['stage07_completion_sha256']==digest(ROOT/'results'/'stage07'/f'{TEST_VERSION}_stage07_complete.json'),'Stage07 SHA link')
features=list(p12['feature_names_ordered']);target=str(p12['target_column'])
require(features==sdev['core_features']==stest['core_features'],'CORE feature order changed')
require(target==sdev['target']==stest['target']=='manual_is_correct','Label schema mismatch')
require(len(features)==8 and target not in features and 'manual_error_type' not in features,'CORE schema leakage')
expected_models=set(p12['model_ids']);require(len(expected_models)==3,'Unexpected model count')

old=load_json(OLD_DEV_PATH)
old_ids={r['question_id'] for r in old}
require(len(old_ids)==len(old)==1000,'Development unique IDs')
require(not old_ids.intersection(selected_ids),'Test/development question overlap')
require(m13['stage03_sanity_questions_retained']==30,'Stage03 sanity count')
print('STAGE 12/13 + NEW STAGE 08 FROZEN SHA: PASS')
print('250 selected test IDs; 50 reserve IDs; old dev IDs disjoint')

STAGE 12/13 + NEW STAGE 08 FROZEN SHA: PASS
250 selected test IDs; 50 reserve IDs; old dev IDs disjoint


## 3 — DEV etiketleriyle 970/model eğitim; TEST matrisinden yalnız predictors yükle

In [4]:
# IMPORTANT: This stage does NOT load test manual_is_correct, old full-new master, or final-test gold.
DEV_MATRIX_PATH=Path(m08dev['outputs']['extended_model_matrix'])
require(digest(DEV_MATRIX_PATH)==p12['stage08_matrix_sha256'],'Frozen DEV extended matrix SHA')
needed_dev=['question_id','model_id','exclude_from_final_test','used_in_model_selection',target]+features
dev_all=pd.read_parquet(DEV_MATRIX_PATH,columns=needed_dev)
require(len(dev_all)==3000 and not dev_all.duplicated(['question_id','model_id']).any(),'Frozen DEV rows/keys')
require(set(dev_all['question_id'])==old_ids,'Frozen DEV IDs differ from old 1000')
train=dev_all.loc[dev_all['exclude_from_final_test'].astype(int)==0].copy()
require(len(train)==2910,'CLEAN_DEV must have 2910 rows')
require((train.groupby('model_id').size()==970).all(),'CLEAN_DEV count per model')
require(not train['used_in_model_selection'].astype(bool).any(),'Stage03 overlap in training')
require(train.groupby('model_id')['question_id'].nunique().eq(970).all(),'CLEAN_DEV distinct IDs')
require(set(train['model_id'])==expected_models,'Train model IDs')
require(set(train[target].dropna().astype(int).unique())=={0,1} and train[target].notna().all(),'Train labels')

# Parquet column projection prevents reading final-test labels during prediction freeze.
test_cols=['question_id','model_id']+features
test_full=pd.read_parquet(TEST_MATRIX_PATH,columns=test_cols)
require(len(test_full)==3750 and not test_full.duplicated(['question_id','model_id']).any(),'New matrix N/unique keys')
require(set(test_full['question_id'])==old_ids|selected_ids,'Inference matrix contains reserve or misses expected IDs')
require(set(test_full['model_id'])==expected_models,'Unexpected new model IDs')
test=test_full.loc[test_full['question_id'].isin(selected_ids)].copy().sort_values(['model_id','question_id'],kind='mergesort').reset_index(drop=True)
require(len(test)==750 and (test.groupby('model_id').size()==250).all(),'Final TEST 750 rows / 250 per model')
require(set(test['question_id'])==selected_ids,'Selected test ID mismatch')
require(not set(test['question_id']).intersection(set(train['question_id'])),'Train/test contamination')
for frame in (train,test):
    for col in features:
        require(np.isfinite(frame[col].to_numpy(dtype=float)).all(),f'Invalid predictor: {col}')
print('FROZEN DEV: 970 × 3 =',len(train),'| FINAL TEST PREDICTOR VIEW:',len(test))
print('TEST gold and manual labels have not been loaded.')

FROZEN DEV: 970 × 3 = 2910 | FINAL TEST PREDICTOR VIEW: 750
TEST gold and manual labels have not been loaded.


## 4 — Yalnız DEV ile Logistic CORE ve XGBoost CORE fit; tahminleri dondur

In [5]:
fit_spec_log=p12['logistic_core']; fit_spec_xgb=p12['xgb_core']
require(fit_spec_log['features']==features==fit_spec_xgb['features'],'Fitted CORE feature contract')
require(fit_spec_log['training_records_per_llm']==970,'Logistic dev count contract')
shared=fit_spec_xgb['shared_parameters'];candidates=fit_spec_xgb['hyperparameter_candidates']
require(len(candidates)==4,'XGBoost candidate grid is not 4')
require(fit_spec_xgb['hyperparameter_selection']['selection_metric']=='mean_validation_brier','Inner metric changed')
require(fit_spec_xgb['hyperparameter_selection']['inner_cv']=='StratifiedKFold(n_splits=3, shuffle=True, random_state=17011)','Inner CV changed')
require(int(p12['evaluation']['secondary_ece']['n_bins'])==10,'ECE bins changed')
model_params=fit_spec_log['params']; C=float(model_params['C'])
require(model_params['solver']=='lbfgs' and int(model_params['max_iter'])==3000 and int(model_params['random_state'])==17,'Logistic contract changed')

# If predictions are already frozen, use exactly those bytes; do not refit or reselect.
if PRED_PATH.exists() or PRED_MANIFEST_PATH.exists():
    require(PRED_PATH.is_file() and PRED_MANIFEST_PATH.is_file(),'Partial frozen prediction outputs')
    pm=load_json(PRED_MANIFEST_PATH)
    require(pm['predictions_sha256']==digest(PRED_PATH),'Previously frozen prediction SHA mismatch')
    for item in pm['provenance']:
        require(digest(item['path'])==item['sha256'],f'Frozen provenance changed: {item["path"]}')
    frozen_preds=pd.read_parquet(PRED_PATH)
    print('PREDICTIONS ALREADY FROZEN — reused without training')
else:
    rows=[]; fit_audit=[]; inner_rows=[]
    for mid in sorted(expected_models):
        d=train.loc[train['model_id']==mid].sort_values('question_id',kind='mergesort').reset_index(drop=True)
        t=test.loc[test['model_id']==mid].sort_values('question_id',kind='mergesort').reset_index(drop=True)
        x=d[features].to_numpy(dtype=float); y=d[target].astype(int).to_numpy()
        xt=t[features].to_numpy(dtype=float)
        require(len(d)==970 and len(t)==250 and len(np.unique(y))==2,'Bad per-model train/test')
        pipe=Pipeline([('scaler',StandardScaler()),('logistic',LogisticRegression(C=C,solver='lbfgs',max_iter=3000,random_state=17))])
        pipe.fit(x,y)
        p_log=pipe.predict_proba(xt)[:,1]
        folds=list(StratifiedKFold(n_splits=3,shuffle=True,random_state=17011).split(x,y))
        candidate_scores=[]
        for candidate in candidates:
            errs=[]
            for j,(tr,va) in enumerate(folds):
                cfg=XGBClassifier(**shared,**candidate)
                cfg.fit(x[tr].astype(np.float32),y[tr],verbose=False)
                pv=cfg.predict_proba(x[va].astype(np.float32))[:,1]
                loss=float(brier_score_loss(y[va],pv))
                errs.append(loss)
                inner_rows.append({'model_id':mid,'max_depth':int(candidate['max_depth']), 'n_estimators':int(candidate['n_estimators']),'inner_fold':j,'n_train':len(tr),'n_valid':len(va),'inner_brier':loss})
            candidate_scores.append((float(np.mean(errs)),int(candidate['n_estimators']),int(candidate['max_depth']),candidate))
        best=min(candidate_scores,key=lambda v:(v[0],v[1],v[2]))
        selected=dict(best[3]); xgb=XGBClassifier(**shared,**selected)
        xgb.fit(x.astype(np.float32),y,verbose=False)
        p_xgb=xgb.predict_proba(xt.astype(np.float32))[:,1]
        p_raw=t['verification_p_correct'].to_numpy(dtype=float)
        p_base=np.full(len(t),float(np.mean(y)),dtype=float)
        for name,vec in [('logistic_core',p_log),('xgb_core',p_xgb),('raw_verification',p_raw),('train_prevalence',p_base)]:
            require(np.isfinite(vec).all() and (vec>=0).all() and (vec<=1).all(),f'Invalid probability {mid}/{name}')
            for qid,p in zip(t['question_id'],vec):
                rows.append({'model_id':mid,'question_id':qid,'method':name,'p_correct':float(p)})
        fit_audit.append({'model_id':mid,'training_n':970,'test_n':250,'dev_label_prevalence':float(np.mean(y)), 'xgb_inner_cv_mean_brier':best[0], 'xgb_selected_max_depth':selected['max_depth'],'xgb_selected_n_estimators':selected['n_estimators']})
        print('FITTED ON DEV ONLY:',mid,'| xgb:',selected)
    frozen_preds=pd.DataFrame(rows).sort_values(['model_id','question_id','method'],kind='mergesort').reset_index(drop=True)
    require(len(frozen_preds)==3000 and not frozen_preds.duplicated(['model_id','question_id','method']).any(),'Predicted rows/keys')
    OUT.mkdir(parents=True,exist_ok=True)
    fit_path=OUT/'development_fit_audit_v1.json'
    inner_path=OUT/'development_xgb_inner_cv_v1.csv'
    fit_sha=freeze_json(fit_path,fit_audit)
    inner_sha=freeze_csv(inner_path,pd.DataFrame(inner_rows))
    pred_sha=freeze_parquet(PRED_PATH,frozen_preds)
    sources=[PROTOCOL_PATH,M12_PATH,M13_PATH,AMEND_PATH,QUESTIONS_PATH,SPLIT_PATH,OLD_DEV_PATH,DEV_M08_PATH,DEV_SCHEMA_PATH,DEV_MATRIX_PATH,TEST_M08_PATH,TEST_SCHEMA_PATH,TEST_MATRIX_PATH,TEST_DATA_PATH,TEST_DATA_MANIFEST_PATH,fit_path,inner_path]
    pm={'stage':'14_independent_test_prediction_freeze','dataset_version':TEST_VERSION,'n_independent_test_questions':250,'n_model_answer_rows':750,'n_method_prediction_rows':3000,'methods':['logistic_core','xgb_core','raw_verification','train_prevalence'],'core_features_ordered':features,'test_labels_used_in_fit_selection_or_prediction':False,'training_n_per_model':970,'predictions_path':str(PRED_PATH),'predictions_sha256':pred_sha,'fit_audit_sha256':fit_sha,'inner_cv_sha256':inner_sha,'provenance':[{'path':str(p),'sha256':digest(p)} for p in sources],'status':'PREDICTIONS_FROZEN_BEFORE_TEST_LABEL_EVALUATION'}
    manifest_sha=freeze_json(PRED_MANIFEST_PATH,pm)

require(len(frozen_preds)==3000 and not frozen_preds.duplicated(['model_id','question_id','method']).any(),'Frozen prediction row keys')
require(set(frozen_preds['method'])=={'logistic_core','xgb_core','raw_verification','train_prevalence'},'Frozen methods')
require(set(frozen_preds['question_id'])==selected_ids,'Frozen selected IDs')
require((frozen_preds.groupby(['model_id','method']).size()==250).all(),'Frozen method count')
require(set(frozen_preds['model_id'])==expected_models,'Frozen model IDs')
require(np.isfinite(frozen_preds['p_correct']).all() and frozen_preds['p_correct'].between(0,1).all(),'Frozen probability range')
print('PREDICTIONS FROZEN:',PRED_PATH)
print('PREDICTIONS SHA:',digest(PRED_PATH))
print('PREDICTION MANIFEST SHA:',digest(PRED_MANIFEST_PATH))
print('EVALUATION SKIPPED: test labels have not been loaded; set RUN_EVALUATION=True ONLY in last cell after reviewing freeze.')

FITTED ON DEV ONLY: Qwen/Qwen3-4B | xgb: {'max_depth': 2, 'n_estimators': 120}
FITTED ON DEV ONLY: google/gemma-3-4b-it | xgb: {'max_depth': 2, 'n_estimators': 120}
FITTED ON DEV ONLY: microsoft/Phi-4-mini-instruct | xgb: {'max_depth': 2, 'n_estimators': 120}
PREDICTIONS FROZEN: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage14/validated_v012_final250_v1/final_test_predictions_v1.parquet
PREDICTIONS SHA: 6b00973b5b867994f0bc26805fea0893021b3f724d97e4b84e4124c7d17ab8fb
PREDICTION MANIFEST SHA: 0fa8b40eae407dec680d07089f23e52b324ca8ac37ffb1d4581569cfa40fa93c
EVALUATION SKIPPED: test labels have not been loaded; set RUN_EVALUATION=True ONLY in last cell after reviewing freeze.


## 5 — Bağımsız test metrikleri (YALNIZCA ikinci çalıştırmada)

İlk çalıştırmayı burada `RUN_EVALUATION=False` ile bitirin. SHA çıktısını kontrol ettikten sonra **yalnız aşağıdaki son hücredeki** `RUN_EVALUATION = True` satırını değiştirerek bu hücreyi çalıştırın. Test etiketleri ancak dondurulmuş tahmin dosyası/provenance teyidinden sonra yüklenir. Brier ana metrik; ikincil NLL, ECE, AUROC, AUPRC, AURC, risk@coverage ve önceden sabit eşikler. Model başına eşleştirilmiş 4000 soruluk bootstrap. Sonuçlara bakarak yöntem/threshold/feature seçmeyin.

In [6]:
RUN_EVALUATION = True  # İlk çalıştırmada False; tahmin freeze kontrolünden SONRA yalnız burada True.
if not RUN_EVALUATION:
    print('EVALUATION SKIPPED: frozen predictions are ready; no final test labels loaded.')
else:
    require(PRED_PATH.is_file() and PRED_MANIFEST_PATH.is_file(),'Freeze predictions first')
    pm=load_json(PRED_MANIFEST_PATH)
    require(digest(PRED_PATH)==pm['predictions_sha256'],'Frozen prediction SHA mismatch')
    for item in pm['provenance']:
        require(digest(item['path'])==item['sha256'],f'Prediction provenance modified: {item["path"]}')
    # Final labels are first loaded here, never in earlier cells.
    label_frame=pd.read_parquet(TEST_MATRIX_PATH,columns=['question_id','model_id',target])
    labels=label_frame.loc[label_frame['question_id'].isin(selected_ids)].copy()
    require(len(labels)==750 and not labels.duplicated(['model_id','question_id']).any(),'Test label keys')
    require(labels[target].notna().all() and labels[target].isin([0,1]).all(),'Unlabeled final answers')
    preds=pd.read_parquet(PRED_PATH)
    joined=preds.merge(labels,on=['model_id','question_id'],how='left',validate='many_to_one',indicator=True)
    require(len(joined)==3000 and (joined['_merge']=='both').all(),'Prediction–label join')
    joined=joined.drop(columns='_merge')
    ev=p12['evaluation']; sel=ev['selective_answering']; uncertainty=p12['uncertainty']
    levels=list(sel['coverage_levels']); thresholds=list(sel['fixed_probability_thresholds'])
    require(levels==[0.2,0.5,0.8,1.0] and thresholds==[0.8,0.9],'Selective policy differs from frozen protocol')
    EPS=float(ev['secondary_nll']['clip_eps']); BINS=int(ev['secondary_ece']['n_bins'])
    require(EPS==1e-6 and BINS==10,'NLL/ECE definition changed')
    B=int(uncertainty['bootstrap_repetitions']); seed=int(uncertainty['bootstrap_seed'])
    require(B==4000 and seed==20260921,'Bootstrap contract changed')

    def metrics(y,p):
        y=np.asarray(y,dtype=int);p=np.asarray(p,dtype=float)
        require(len(y)==250 and set(np.unique(y)).issubset({0,1}),'Unexpected final n/labels')
        conf=np.minimum((np.clip(p,0.,1.)*BINS).astype(int),BINS-1)
        ece=0.
        for bin_id in range(BINS):
            idx=conf==bin_id
            if idx.any(): ece+=(idx.mean())*abs(float(y[idx].mean())-float(p[idx].mean()))
        safe=np.clip(p,EPS,1.-EPS)
        return {'n':len(y),'prevalence':float(y.mean()), 'brier':float(np.mean((p-y)**2)),
                'nll':float(-np.mean(y*np.log(safe)+(1-y)*np.log1p(-safe))),
                'ece':float(ece),
                'auroc':float(roc_auc_score(y,p)) if len(np.unique(y))==2 else np.nan,
                'auprc':float(average_precision_score(y,p)) if len(np.unique(y))==2 else np.nan}

    def riskcurve(question_ids,y,p):
        # Deterministic tie-break: decreasing score, increasing question_id.
        order=np.lexsort((np.asarray(question_ids,dtype=str),-np.asarray(p,dtype=float)))
        wrong=(1-np.asarray(y,dtype=int))[order]
        cumul=np.cumsum(wrong)/np.arange(1,len(wrong)+1)
        return float(cumul.mean()),cumul

    metric_rows=[];selective_rows=[];boot_rows=[]
    for mid, group in joined.groupby('model_id',sort=True):
        wide_p=group.pivot(index='question_id',columns='method',values='p_correct').sort_index()
        wide_y=group[['question_id',target]].drop_duplicates('question_id').set_index('question_id').sort_index()
        require(len(wide_p)==len(wide_y)==250 and wide_p.index.equals(wide_y.index),'Paired question ordering')
        ids=wide_p.index.to_numpy(dtype=str)
        y=wide_y[target].to_numpy(dtype=int)
        methods=list(pm['methods'])
        for method in methods:
            p=wide_p[method].to_numpy(dtype=float)
            metric_rows.append({'model_id':mid,'method':method,**metrics(y,p),'aurc':riskcurve(ids,y,p)[0]})
            aurc,curve=riskcurve(ids,y,p)
            for coverage in levels:
                k=max(1,int(math.ceil(coverage*len(y))))
                selective_rows.append({'model_id':mid,'method':method,'policy':'top_k','requested_coverage':coverage,'threshold':np.nan,'n_selected':k,'attained_coverage':k/len(y),'empirical_risk':float(curve[k-1]),'aurc':aurc})
            if method in ('logistic_core','xgb_core'):
                for threshold in thresholds:
                    mask=p>=threshold
                    selective_rows.append({'model_id':mid,'method':method,'policy':'fixed_threshold','requested_coverage':np.nan,'threshold':threshold,'n_selected':int(mask.sum()),'attained_coverage':float(mask.mean()),'empirical_risk':float(np.mean(1-y[mask])) if mask.any() else np.nan,'aurc':np.nan})
        # Primary paired Brier: XGB minus Logistic on the same 250 questions.
        px=wide_p['xgb_core'].to_numpy(dtype=float);pl=wide_p['logistic_core'].to_numpy(dtype=float)
        lossdiff=(px-y)**2-(pl-y)**2
        require(len(lossdiff)==250,'Primary paired sample count')
        rng=np.random.default_rng(seed)
        sample_indices=rng.integers(0,250,size=(B,250),endpoint=False)
        paired_means=lossdiff[sample_indices].mean(axis=1)
        ci=np.quantile(paired_means,[0.025,0.975])
        boot_rows.append({'model_id':mid,'comparison':'brier_xgb_minus_logistic','n_independent_questions':250,'n_bootstrap':B,'effect':float(lossdiff.mean()),'ci95_low':float(ci[0]),'ci95_high':float(ci[1]),'interval':'paired_question_percentile_95','interpretation':'descriptive; lower Brier corresponds to negative difference'})
        # Exploratory paired AURC and risk@coverage intervals on same resampled questions.
        delta_aurc=[]; delta_risk={c:[] for c in levels}
        for sample in sample_indices:
            sy=y[sample]; sid=ids[sample]; sx=px[sample]; sl=pl[sample]
            ax,cx=riskcurve(sid,sy,sx); al,cl=riskcurve(sid,sy,sl)
            delta_aurc.append(ax-al)
            for c in levels:
                k=max(1,int(math.ceil(c*len(sample))))
                delta_risk[c].append(float(cx[k-1]-cl[k-1]))
        for label,values in [('aurc_xgb_minus_logistic',delta_aurc)]+[(f'risk_at_coverage_{c:g}_xgb_minus_logistic',delta_risk[c]) for c in levels]:
            arr=np.asarray(values,dtype=float);q=np.quantile(arr,[0.025,0.975])
            effect=(riskcurve(ids,y,px)[0]-riskcurve(ids,y,pl)[0]) if label.startswith('aurc') else (riskcurve(ids,y,px)[1][max(1,int(math.ceil(float(label.split('_')[3])*250)))-1]-riskcurve(ids,y,pl)[1][max(1,int(math.ceil(float(label.split('_')[3])*250)))-1])
            boot_rows.append({'model_id':mid,'comparison':label,'n_independent_questions':250,'n_bootstrap':B,'effect':float(effect),'ci95_low':float(q[0]),'ci95_high':float(q[1]),'interval':'exploratory_paired_question_percentile_95','interpretation':'exploratory'})
        print('EVALUATED:',mid,'| paired questions:',len(y))
    met=pd.DataFrame(metric_rows).sort_values(['model_id','method']).reset_index(drop=True)
    selective=pd.DataFrame(selective_rows).sort_values(['model_id','method','policy','requested_coverage','threshold']).reset_index(drop=True)
    boot=pd.DataFrame(boot_rows).sort_values(['model_id','comparison']).reset_index(drop=True)
    require(len(met)==12 and len(selective)==3*(4*4+2*2) and len(boot)==3*(1+1+4),'Final output rows')
    OUT.mkdir(parents=True,exist_ok=True)
    msha=freeze_csv(EVAL_METRICS_PATH,met)
    ssha=freeze_csv(EVAL_SELECTIVE_PATH,selective)
    bsha=freeze_csv(EVAL_BOOTSTRAP_PATH,boot)
    evalu_manifest={'stage':'14_independent_final_test_evaluation','source_predictions_sha256':digest(PRED_PATH),'source_prediction_manifest_sha256':digest(PRED_MANIFEST_PATH),'stage07_labels_manifest_sha256':digest(ROOT/'results'/'stage07'/f'{TEST_VERSION}_stage07_complete.json'),'stage08_core_matrix_sha256':digest(TEST_MATRIX_PATH),'final_test_questions_sha256':digest(QUESTIONS_PATH),'n_unique_final_questions_per_model':250,'n_final_model_answer_rows':750,'n_method_prediction_rows':3000,'primary_comparison':'Brier xgb_core minus logistic_core','paired_bootstrap_B':B,'paired_bootstrap_seed':seed,'metric_csv_sha256':msha,'selective_csv_sha256':ssha,'bootstrap_csv_sha256':bsha,'metric_csv_path':str(EVAL_METRICS_PATH),'selective_csv_path':str(EVAL_SELECTIVE_PATH),'bootstrap_csv_path':str(EVAL_BOOTSTRAP_PATH),'difficulty_amendment_sha256':digest(AMEND_PATH),'test_labels_used_only_for_independent_evaluation':True,'status':'FINAL_TEST_EVALUATED'}
    esha=freeze_json(EVAL_MANIFEST_PATH,evalu_manifest)
    print('STAGE14 INDEPENDENT FINAL TEST COMPLETE')
    print('Metrics:',EVAL_METRICS_PATH,'SHA:',msha)
    print('Selective:',EVAL_SELECTIVE_PATH,'SHA:',ssha)
    print('Paired bootstrap:',EVAL_BOOTSTRAP_PATH,'SHA:',bsha)
    print('Evaluation manifest:',EVAL_MANIFEST_PATH,'SHA:',esha)
    display(met[['model_id','method','n','brier','nll','ece','auroc','auprc','aurc']])

EVALUATED: Qwen/Qwen3-4B | paired questions: 250
EVALUATED: google/gemma-3-4b-it | paired questions: 250
EVALUATED: microsoft/Phi-4-mini-instruct | paired questions: 250
STAGE14 INDEPENDENT FINAL TEST COMPLETE
Metrics: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage14/validated_v012_final250_v1/final_test_metrics_v1.csv SHA: 3f582f58327d9a84f7da6aea3427ce5b149b2d90a4b31a1a2e89b68afb8800b7
Selective: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage14/validated_v012_final250_v1/final_test_selective_v1.csv SHA: bc86e1bf68f03c91d184e39c0517b704fdb7b1960900e98c29cb2f34081a3d71
Paired bootstrap: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage14/validated_v012_final250_v1/final_test_paired_bootstrap_v1.csv SHA: 6094181c73577dd938a1ecad9499cf1a9fa4a2d9efd6e49c774fefc5b6dac116
Evaluation manifest: /content/drive/MyDrive/LLM_Confidence_Thesis/results/stage14/validated_v012_final250_v1/stage14_evaluation_manifest_v1.json SHA: a68baf340eda3092864cb17ec027675ac8f90e6

,model_id,method,n,brier,nll,ece,auroc,auprc,aurc
0,Qwen/Qwen3-4B,logistic_core,250,0.109988,0.346982,0.054721,0.928543,0.901839,0.266626
1,Qwen/Qwen3-4B,raw_verification,250,0.160299,1.426542,0.162172,0.900751,0.878260,0.280206
2,Qwen/Qwen3-4B,train_prevalence,250,0.245761,0.684659,0.053072,0.500000,0.416000,0.604697
3,Qwen/Qwen3-4B,xgb_core,250,0.095669,0.314382,0.088154,0.941023,0.916300,0.259226
4,google/gemma-3-4b-it,logistic_core,250,0.161529,0.489461,0.047151,0.820376,0.871576,0.171212
5,google/gemma-3-4b-it,raw_verification,250,0.311784,2.599503,0.313754,0.766162,0.834472,0.200066
6,google/gemma-3-4b-it,train_prevalence,250,0.235832,0.664719,0.034763,0.500000,0.624000,0.364519
7,google/gemma-3-4b-it,xgb_core,250,0.149026,0.455896,0.052068,0.853996,0.901005,0.150513
8,microsoft/Phi-4-mini-instruct,logistic_core,250,0.139843,0.420811,0.068166,0.845282,0.656922,0.502570
9,microsoft/Phi-4-mini-instruct,raw_verification,250,0.262524,0.999763,0.252986,0.794511,0.534126,0.556587
